# 01. 산출물 계약과 승인 게이트

목표: `intent → spec → plan`이 단순 문서 목록이 아니라 다음 단계가 읽을 수 있는 계약임을 구현합니다. Python 표준 라이브러리만 사용하며 위에서 아래로 실행합니다.

In [ ]:
from dataclasses import dataclass, field
from datetime import datetime, timezone
from typing import Dict, List

REQUIRED = {
    'intent': {'problem', 'outcome', 'users', 'constraints', 'open_questions'},
    'spec': {'requirements', 'design', 'risks', 'policy_checks'},
    'plan': {'files', 'steps', 'tests', 'rollback'},
}

@dataclass
class Artifact:
    kind: str
    fields: Dict[str, object]
    author: str
    approved_by: str | None = None
    created_at: str = field(default_factory=lambda: datetime.now(timezone.utc).isoformat())

    def problems(self) -> List[str]:
        missing = REQUIRED[self.kind] - self.fields.keys()
        empty = {key for key, value in self.fields.items() if value in ('', [], None)}
        return [f'누락: {name}' for name in sorted(missing)] + [f'비어 있음: {name}' for name in sorted(empty)]

    @property
    def accepted(self) -> bool:
        return not self.problems() and self.approved_by is not None


## 의도 문서 검증

형식 검증은 내용의 옳음을 보장하지 않습니다. 하지만 다음 단계가 필요한 정보 없이 시작되는 것을 결정론적으로 막을 수 있습니다.

In [ ]:
intent = Artifact('intent', {
    'problem': '고객이 배포 상태를 알 수 없다',
    'outcome': '포털에서 단계와 예상 시간을 확인한다',
    'users': ['고객', '운영 상담원'],
    'constraints': ['새 개인정보를 세션에 저장하지 않는다'],
    'open_questions': ['외부 파트너도 접근해야 하는가?'],
}, author='product-owner')

print('승인 전:', intent.accepted, intent.problems())
intent.approved_by = 'service-owner'
print('승인 후:', intent.accepted)
assert intent.accepted


In [ ]:
def may_start(next_stage: str, artifacts: Dict[str, Artifact]) -> tuple[bool, str]:
    prerequisite = {'design': 'intent', 'build': 'spec', 'test': 'plan'}[next_stage]
    artifact = artifacts.get(prerequisite)
    if artifact is None:
        return False, f'{prerequisite} 산출물이 없습니다'
    if not artifact.accepted:
        return False, f'{prerequisite}가 완전하지 않거나 승인되지 않았습니다'
    return True, f'{next_stage} 시작 가능'

allowed, reason = may_start('design', {'intent': intent})
print(allowed, reason)
assert allowed


## 확장 과제

1. `spec`과 `plan`을 만들고 세 게이트를 모두 통과시키세요.
2. 위험 등급이 높을 때 별도 정책 소유자 승인을 요구하세요.
3. 각 산출물에 앞 단계의 commit SHA를 넣어 추적성을 검사하세요.